# DiagnosticIQ Verbose

Paraphrases each DiagnosticIQ question's symbolic conditions into natural language (paper Appendix A.6.3).

1. **Representative examples.** Embed every question's conditions with `all-mpnet-base-v2`, group them by cosine similarity and sample 10 representative questions. Their ids are saved to `verbose/incontext_expl.json`.
2. **Manual paraphrases.** The conditions of those 10 questions are rewritten in natural language by hand and stored as `verbose/condition_verberlizer_examples.txt`. This step happens outside the notebook.
3. **LLM paraphrasing.** `mistralai/mistral-large` on watsonx.ai is prompted with those examples to paraphrase the remaining questions. The result is saved as `datasets/DiagnosticIQVerbose`.

Step 3 needs `WATSONX_APIKEY`, `WATSONX_URL` and `WATSONX_PROJECT_ID` in the environment or the repository `.env`.

In [ ]:
import os, sys
import re
import json
from datetime import datetime
from typing import Optional, List, Callable, Any, Tuple, Dict
from abc import abstractmethod, ABC
import random
import numpy as np
import pandas as pd
import copy
import pickle
from tqdm.autonotebook import tqdm
import itertools
from dotenv import load_dotenv

load_dotenv()

random.seed(42)
np.random.seed(42)


In [ ]:
from dataset_utils.paths import get_dataset_dir
from dataset_utils.reader import ADIQDataset
from dataset_utils.outputs import to_basic_prompt, question_formatting
from dataset_utils.utils import file_handle, grouping
from dataset_utils.utils.concurrency import concurrent_dict_execution

dataset_dir = get_dataset_dir()
generated_datasets_dir = dataset_dir / "datasets"
verbose_dir = dataset_dir / "verbose"
os.makedirs(verbose_dir, exist_ok=True)

# Paraphrase the base DiagnosticIQ questions.
source_dataset = "DiagnosticIQ"

## 1. Representative examples

In [ ]:
ds = ADIQDataset(generated_datasets_dir / source_dataset)

In [ ]:
conditions_of_questions = {x.id:x.condition_description for x in ds.questions}
conditions_of_questions = {k:"\n".join(v) for k,v in conditions_of_questions.items()}

In [ ]:
from sentence_transformers import SentenceTransformer, util


model = SentenceTransformer("all-mpnet-base-v2")

In [ ]:
enc = model.encode(list(conditions_of_questions.values()))
qcond_index_map = {k:v for v,k in enumerate(conditions_of_questions.keys())}
index_qcond_map = {v:k for k,v in qcond_index_map.items()}

norm = np.linalg.norm(enc, axis=1)
norm = np.outer(norm,norm) + 1e-6

cosine_sim = (enc @ enc.T)/norm
cosine_sim = np.abs(cosine_sim)

In [ ]:
for r in range(985,1000):
    _thresh = r/1000

    _groups = {}
    for v in range(cosine_sim.shape[0]):
        thresh = np.quantile(cosine_sim[v].flatten(),_thresh)
        _groups[v] = set(np.where(cosine_sim[v]>thresh)[0].tolist())-{v}


    groups = {}
    for k,v in _groups.items():
        groups[index_qcond_map[k]] = set([index_qcond_map[x] for x in v])

    edges = set()
    for k,v in tqdm(groups.items()):
        for n in v:
            if k != n: 
                n_min,n_max = min(k,n), max(k,n)
                edges.add((n_min, n_max))

    group_set = grouping.getComponents(len(groups.keys()), edges)

    print("r:{} , len({})".format(_thresh, len(group_set)))


In [ ]:
_thresh = 0.988

_groups = {}
for v in range(cosine_sim.shape[0]):
    thresh = np.quantile(cosine_sim[v].flatten(),_thresh)
    _groups[v] = set(np.where(cosine_sim[v]>thresh)[0].tolist())-{v}


groups = {}
for k,v in _groups.items():
    groups[index_qcond_map[k]] = set([index_qcond_map[x] for x in v])

edges = set()
for k,v in tqdm(groups.items()):
    for n in v:
        if k != n: 
            n_min,n_max = min(k,n), max(k,n)
            edges.add((n_min, n_max))

group_set = grouping.getComponents(len(groups.keys()), edges)

In [ ]:
incontext_expl = random.sample([random.choice(x) for x in group_set], 10)

# Overwrites the saved example ids; skip this cell to keep the existing ones.
file_handle.save_json(
    {"example_id":incontext_expl},
    verbose_dir / "incontext_expl.json"
)

## 2. Manual paraphrases

Rewrite the conditions of the questions listed in `verbose/incontext_expl.json` in natural language and save them, in the prompt format used below, as `verbose/condition_verberlizer_examples.txt`.

## 3. LLM paraphrasing

In [ ]:
ds = ADIQDataset(generated_datasets_dir / source_dataset)

incontext_expl = file_handle.load_json(
    verbose_dir / "incontext_expl.json"
)['example_id']


In [ ]:
question_template = """
## Asset Description:
{asset_type}: {asset_description}

## Conditions:
{conditions}

## Conditions in Natural Language:
"""

main_template = """
Your task is to read the asset description (## Asset Description:) and conditions (## Conditions:) applied on the asset and 
write the conditions (## Conditions:) in natural language  several examples are provided complete the last last sample.

{examples}
{question}
"""


In [ ]:
from dataset_utils.question import Question

def condition_list(q:Question) -> str:
    qlist = question_template.format(
        asset_type = q.asset_type,
        asset_description = ds.asset_descriptions[q.asset_type],
        conditions = "\n".join(q.condition_description)
    )

    return qlist

def verberlizer_incontext(q:Question, incontext_expl:str) -> str:
    _q = question_template.format(
        asset_type = q.asset_type,
        asset_description = ds.asset_descriptions.get(q.asset_type,"NONE"),
        conditions = "\n".join(q.condition_description)
    )

    _template = main_template.format(
        examples = incontext_expl,
        question = _q
    )

    return _template


In [ ]:
import dataset_utils.watsonx as watsonx

_examples = file_handle.load_text(verbose_dir / "condition_verberlizer_examples.txt")

def _verbalize_conditions(q:Question) -> str:
    _prompt = verberlizer_incontext(q,_examples)
    response = watsonx.get_completion_response(
        _prompt
    )

    return watsonx.clean_response(response)


In [ ]:
param_dict = {k:[v] for k,v in enumerate(ds.questions)}

for k,v in concurrent_dict_execution(
    _verbalize_conditions,
    param_dict
):
    ds.questions[k].verberlized_conditions = v


## Question formatting

Builds the final question text with the paraphrased conditions in place of the symbolic list, as the DiagnosticIQ Verbose evaluation prompt does.

In [ ]:
missing = [q.id for q in ds.questions if q.verberlized_conditions is None]
if missing:
    raise RuntimeError(f"{len(missing)} questions have no paraphrase; re-run them before saving: {missing[:10]}")

for q in ds.questions:
    question_formatting(q, ds.asset_descriptions, condition_str=q.verberlized_conditions)

In [ ]:
ds.save(
    "DiagnosticIQVerbose",
    generated_datasets_dir
)